# 02 — Feature Engineering

Builds the feature table and inspects the leakage checks. Every feature is
computed within its own `mandi + variety + grade` group from values at or
before the row's own date.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 60)


In [ ]:
from src import config, features, preprocessing, targets, validation

bundle = preprocessing.load_bundle()
df = bundle['data']
feature_cols = bundle['feature_columns']
print(f'{len(df):,} rows, {len(feature_cols)} features')

## Feature families

The experiment ladder in notebook 03 adds these one family at a time.

In [ ]:
for name in features.FEATURE_SETS:
    selected = features.select_feature_set(feature_cols, name)
    print(f'{name:10s} {len(selected):>4} features')

## Target coverage

A target exists only where the group has an observation exactly `h` calendar days later.

In [ ]:
targets.target_coverage(df)

## Leakage checks

The empirical probe recomputes every feature from history truncated at the
row's own date. Any feature that changes value was built with a
forward-looking window.

In [ ]:
structural = validation.structural_leakage_checks(feature_cols, df)
print('structural:', structural)

empirical = validation.empirical_leakage_probe(df, feature_cols, n_probe=20)
print('empirical :', {k: v for k, v in empirical.items() if k != 'leaky_features'})
print('leaky     :', empirical['leaky_features'])

## Missingness introduced by the lookback

The longest window is 28 observations, so early rows in each group have undefined history.

In [ ]:
na_share = df[feature_cols].isna().mean().sort_values(ascending=False)
na_share[na_share > 0].head(20).round(4)